# 15 · Choosing a serving engine
Level L3 · Part 2 · core · ~20 min · builds on 13

## Goal
By the end of this lesson you can compare vLLM, SGLang, TensorRT-LLM, TGI, Triton, llama.cpp and Ollama on the questions that decide a choice, read the defaults of the pinned vLLM image, and pick an engine for a workload with a written reason.

## Why it matters
- **Without a rule:** a team picks the engine it read about last month, then learns late the GPU is unsupported, the engine is in maintenance mode, or a laptop cannot take 300 chats.
- **Without a common API:** each engine has its own client, so every app is rewritten on a switch. Ten apps and four engines is forty integrations.
- **With them:** five questions (hardware, users, prefix, fixed model, what you run) leave one or two engines. Most speak the OpenAI-compatible API, so only the base URL and model name change.
- **When you don't need the comparison:** one person on a laptop with Ollama, or a team whose engine already meets its latency target.

## The idea
Engines are vehicles, not brands. Each one is built for a road. vLLM is the general-purpose van: many models, many GPU types, and one OpenAI-compatible door. SGLang is a van with a shelf for reused packing, strong when many requests share a prefix. TensorRT-LLM is a van built for one exact route: compiled for a fixed model on NVIDIA GPUs, fastest per trip, but you rebuild it for each new route. llama.cpp and Ollama are the bicycle, one rider on a laptop. Pick by road first, then measure the speed on your own load.

## Picture
```mermaid
flowchart TD
  A["Workload facts"] --> B{"One user or no GPU?"}
  B -->|yes| L["llama.cpp or Ollama"]
  B -->|no| C{"Already on TGI?"}
  C -->|yes| M["Plan a move: TGI is in maintenance mode"]
  C -->|no| D{"Already run Triton for other models?"}
  D -->|yes| T1["Triton with an LLM backend"]
  D -->|no| E{"Fixed model on NVIDIA, peak speed wanted?"}
  E -->|yes| T2["TensorRT-LLM, after a vLLM baseline"]
  E -->|no| F{"Many requests share a long prefix?"}
  F -->|yes| S["vLLM and SGLang: benchmark both"]
  F -->|no| V["vLLM"]
```

## Step by step
### Step 1: write down the workload facts
Before you compare engines, write five facts about the workload. The Brightside desk is one example. Fill in your own version in your head or in a notes file.

| Question | Brightside desk (this course) |
|---|---|
| Where does it run? | a GPU VM (L4 class) for the team, a laptop for a demo |
| How many users at once? | hundreds of agents, plus one-off demos |
| Is a prefix shared? | yes: a 1,500-token policy prompt on every request |
| Is the model fixed? | no, it may change every quarter |
| What do you already run? | nothing yet |

### Step 2: compare the engines on the same questions
The tables give the facts that decide the choice, checked on 2026-10-21 against each project's docs. The last column is the CLI equivalent.

**Engines for a GPU team**

| Engine | Hardware | Strength | Watch out for | Start (CLI equivalent) |
|---|---|---|---|---|
| vLLM | NVIDIA, AMD and CPU images | general server, OpenAI API, PagedAttention, continuous batching | memory flags need tuning (lessons 16 and 27) | `vllm serve <model> --port 8000` |
| SGLang | NVIDIA GPUs | RadixAttention prefix reuse, structured and agentic calls | fast-moving; benchmark on your model | `python -m sglang.launch_server --model-path <model> --port 30000` |
| TensorRT-LLM | NVIDIA only | compiled engines, peak speed for a fixed model | a build step per model and GPU; use the NGC container | build an engine, then serve it (NGC docs) |


**Engines you meet in an existing stack**

| Engine | Hardware | Strength | Watch out for | Start (CLI equivalent) |
|---|---|---|---|---|
| TGI | GPU | older Hugging Face server | **maintenance mode** (Hugging Face docs, 2026): no new deployments | keep an existing one running and plan the move |
| Triton Inference Server | NVIDIA | one server for many model types; LLM engines run as backends | LLM serving is a backend (vLLM or TensorRT-LLM); read the notes of the container you pin (v2.73.0, NGC 26.09) | started from a model repository |

**Engines for one person**

| Engine | Hardware | Strength | Watch out for | Start (CLI equivalent) |
|---|---|---|---|---|
| llama.cpp (`llama-server`) | CPU, Apple Metal, CUDA, Vulkan and more | GGUF files, one binary, laptops and small servers | one user or a few; not a GPU fleet | `llama-server --model model.gguf --port 8080` |
| Ollama | laptops and small servers | pull and run a GGUF model with one command; local API on port 11434 | not a fleet engine; check its concurrency limits | `ollama run <model>`, API at `http://localhost:11434/v1` |

**Read it this way:** the first column is the decision. TGI is out for new work, TensorRT-LLM is in only for a fixed NVIDIA model, and llama.cpp and Ollama are for one person or a few.

### Step 3: read the defaults of the pinned image
The pinned vLLM CPU image (`vllm/vllm-openai-cpu:v0.32.0`, the same tag as lessons 08 to 14) has the same `vllm serve` command as a GPU server. Ask it for its flags, and keep only the lines for the port, the host and the API key. This needs Docker. Without Docker the cell below loads the recorded output.

**You should see:** `--port PORT ... (default: 8000)`, and a host flag whose default is `None`.

In [1]:
import subprocess
from pathlib import Path

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"       # the pinned CPU image, same tag as lessons 08 to 14
Path("../data/15").mkdir(parents=True, exist_ok=True)
run = subprocess.run(["docker", "run", "--rm", "--entrypoint", "vllm", IMAGE, "serve", "--help=all"],
                     capture_output=True, text=True, timeout=600)
KEEP = ("--port PORT", "--host HOST", "--api-key", "--served-model-name")
lines = [l.rstrip() for l in run.stdout.splitlines() if any(k in l for k in KEEP)]
if run.returncode == 0 and lines:
    Path("../data/15/serve_flags.txt").write_text("\n".join(lines) + "\n")
    print("\n".join(lines))
else:
    print("docker run failed:", run.stderr[-300:])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**If the cell above was skipped**, this cell prints the recorded output from the same image, so the next steps still run.

In [2]:
import json
from pathlib import Path

live = Path("../data/15/serve_flags.txt")
if live.exists():
    print("source: saved by an earlier run on the pinned image")
    print(live.read_text())
else:
    rec = json.loads(Path("../data/recorded/15.json").read_text())
    print("source:", rec["note"])
    print(rec["stdout"])

source: saved by an earlier run on the pinned image
  --api-key API_KEY [API_KEY ...]
  --host HOST           Host name. (default: None)
  --port PORT           Port number. (default: 8000)
  --served-model-name SERVED_MODEL_NAME [SERVED_MODEL_NAME ...]



### Step 4: one client for every engine
An engine that speaks the OpenAI API is called the same way. Only the base URL and the model name change. `tut_env` is this course's helper that reads your `.env` file. Set `VLLM_BASE_URL` in `.env` to use a server you already run. Without it, the client points at the default local address. The cell prints where the address comes from, never its value.

**You should see:** `base_url: from .env` when `VLLM_BASE_URL` is set in your `.env` (the example `.env` sets it, so this is what you see there). `base_url: default http://localhost:8000/v1` means it is not set. Either way the client is ready. No request is sent yet.

In [3]:
import sys; sys.path.insert(0, "..")   # the course folder, so tut_env.py can be imported
from openai import OpenAI
from tut_env import env

base_url = env("VLLM_BASE_URL", "http://localhost:8000/v1")   # any OpenAI-compatible server
client = OpenAI(base_url=base_url, api_key=env("VLLM_API_KEY", "not-needed"))   # local servers ignore it
print("base_url:", "from .env" if env("VLLM_BASE_URL", "") else "default http://localhost:8000/v1")
print("client ready, model names come from the server's /v1/models")
# To move to Ollama, set base_url to http://localhost:11434/v1 and use its model name. Only the base URL and the model name change.

base_url: from .env
client ready, model names come from the server's /v1/models


### Step 5: apply the rule
The function below follows the picture above, in the same order. It is a rule of thumb, not a benchmark. It returns the engine to test first and the reason.

Its arguments: `gpu` is `none`, `nvidia` or `other`. `users` is `one` or `many`. `shared_prefix` is `low` (the default) or `high`, so pass `"high"` when many requests share a long prefix. `fixed_model` is `True` or `False`. `existing` is `tgi`, `triton` or left out. SGLang's AMD support must be checked on its own hardware page before you use it on an AMD GPU.

In [4]:
def pick_engine(gpu, users, shared_prefix="low", fixed_model=False, existing=None):
    """Rule of thumb from this lesson's picture. gpu: none, nvidia or other."""
    if gpu not in ("none", "nvidia", "other"):
        raise ValueError(f"gpu must be none, nvidia or other; got {gpu!r}")
    if users == "one" or gpu == "none":
        return "llama.cpp or Ollama", "one user or no GPU: GGUF runs on CPU or Metal"
    if existing == "tgi":
        return "plan a move off TGI", "TGI is in maintenance mode; move to vLLM or SGLang"
    if existing == "triton":
        return "Triton with an LLM backend", "one server to operate for many model types"
    if gpu == "nvidia" and fixed_model:
        return "TensorRT-LLM, after a vLLM baseline", "fixed model on NVIDIA: compiled engine for peak speed"
    if shared_prefix == "high":
        return "vLLM and SGLang, benchmark both", "a shared prefix rewards prefix reuse; measure on your traffic"
    return "vLLM", "general-purpose server: OpenAI API, most models and hardware"

Run the rule on five workloads.

In [5]:
scenarios = {
    "Brightside demo on a laptop": dict(gpu="none", users="one"),
    "Desk of 300 agents on one L4 GPU": dict(gpu="nvidia", users="many", shared_prefix="high"),
    "Fixed 8B model on an NVIDIA fleet": dict(gpu="nvidia", users="many", fixed_model=True),
    "Old TGI service": dict(gpu="nvidia", users="many", existing="tgi"),
    "Triton already runs vision models": dict(gpu="nvidia", users="many", existing="triton"),
}
for name, facts in scenarios.items():
    engine, why = pick_engine(**facts)
    print(f"{name}: {engine}\n    because {why}")

Brightside demo on a laptop: llama.cpp or Ollama
    because one user or no GPU: GGUF runs on CPU or Metal
Desk of 300 agents on one L4 GPU: vLLM and SGLang, benchmark both
    because a shared prefix rewards prefix reuse; measure on your traffic
Fixed 8B model on an NVIDIA fleet: TensorRT-LLM, after a vLLM baseline
    because fixed model on NVIDIA: compiled engine for peak speed
Old TGI service: plan a move off TGI
    because TGI is in maintenance mode; move to vLLM or SGLang
Triton already runs vision models: Triton with an LLM backend
    because one server to operate for many model types


### Step 6: check on your own load
The rule gives a first guess only. The real test is your own traffic: the same prompts, the same lengths, the same concurrency, measured on the hardware you will buy. Lesson 27 runs that test with `vllm bench serve`.

## At work
- **TGI is in maintenance mode.** Hugging Face now recommends vLLM, SGLang, llama.cpp and MLX for new serving work, so a TGI pick today is a migration later. → lesson 32 covers deploying the chosen server.
- **Teams route by cache, not by engine name.** Cloudflare raised its cache hit ratio from 60% to 80% with session affinity, and Meta reports 40 to 50% prefix hits with sticky routing. The engine matters less than where the request lands. → lesson 30.
- **Measure on your own load.** Databricks advises benchmarking your own deployment. Anthropic's postmortem: a routing change sent short requests to servers built for long ones, and standard benchmarks missed it. → lesson 27.

## Common mistakes
**1. Treating the port as the same for every engine.** vLLM listens on 8000 by default; the SGLang example above uses 30000. A client that hard-codes the wrong port gets a connection error: the port is the one the server was started with, not the engine's name.

In [6]:
import httpx
httpx.get("http://localhost:8159/health", timeout=2).raise_for_status()   # ConnectError: nothing listens on 8159 here

ConnectError: [Errno 111] Connection refused

**Fix:** read the port from the start command or the startup log, and put it in `.env`.

**2. Typing the hardware name wrong.** The rule accepts only `none`, `nvidia` or `other`. A typo stops it, which is the point: a silent guess would pick the wrong engine.

In [7]:
pick_engine(gpu="NVIDIA", users="many")   # ValueError: the value must be one of the three names

ValueError: gpu must be none, nvidia or other; got 'NVIDIA'

**3. Reading "supports the hardware" as "fast on it".** An engine that runs on a CPU image proves it works, not how fast it is per dollar. Measure on the GPU you plan to buy.

## Try it
The Brightside desk moves to an AMD GPU VM (`gpu="other"`) with 300 users. Its policy prompt is a 1,500-token prefix shared by every request, so pass `shared_prefix="high"`. Predict which engine the rule picks, then run the cell. Then name one thing you would check before you trust the answer.

In [8]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [9]:
engine, why = pick_engine(gpu="other", users="many", shared_prefix="high")
print(engine, "|", why)
# The shared prefix sends the rule to "vLLM and SGLang, benchmark both". TensorRT-LLM is NVIDIA-only, so it is out.
# Without shared_prefix="high" the default "low" gives plain vLLM, which hides the prefix question.
# Before you trust it: check SGLang's supported-hardware page for your exact AMD GPU, and run the lesson 27 load test on that VM.

vLLM and SGLang, benchmark both | a shared prefix rewards prefix reuse; measure on your traffic


## Recap
- Engine follows fit: vLLM default, SGLang when prefixes repeat, TensorRT-LLM for a fixed NVIDIA model, llama.cpp/Ollama for one user or a few; TGI is in maintenance mode.
- Most engines speak the OpenAI API, so the client changes only in the base URL and model name.
- The first choice is a guess until the same load test runs on the hardware you will buy.

**Interview:** "Which engine for a 7B model serving 300 concurrent chats on one L4?" Strong answer: vLLM, for continuous batching and an OpenAI-compatible API; test SGLang if the prompts share a long prefix, and decide on a load test at the target latency.